In [234]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,mean_absolute_error,mean_squared_error,r2_score


## Configuration and constants

In [235]:
RANDOM_STATE=42
ROOT_DIR = Path().cwd().parent
DATASET_DIR = ROOT_DIR / "dataset"
CSV_FILE_PATH = Path.joinpath(DATASET_DIR,"processed","clean_athlete_events.csv")
print(ROOT_DIR,"\n",DATASET_DIR,"\n",CSV_FILE_PATH)

/home/mdev/Documents/ml/olympic_medals_prediction 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset/processed/clean_athlete_events.csv


In [236]:
df = pd.read_csv(CSV_FILE_PATH)
df = df.drop(columns=['Unnamed: 0'])
df.head()

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
0,CHN,1,1932,1,0.000000,22.0,175.0,70.0,0.0,0.0,1,2,0
1,CHN,1,1936,54,0.037037,24.0,175.0,69.0,0.0,0.0,7,27,0
2,CHN,1,1948,31,0.032258,26.0,175.0,70.0,0.0,0.0,6,12,0
3,CHN,1,1952,1,0.000000,23.0,175.0,70.0,0.0,0.0,1,1,0
4,CHN,1,1984,215,0.386047,23.0,173.0,66.0,0.0,0.0,19,105,32


In [237]:
df.describe()

,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
count,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000,3837.000000
mean,0.732343,1981.898879,48.853792,0.217977,25.526192,174.764399,70.297628,4.207975,1.378681,6.980975,29.816523,4.927026
std,0.442795,28.381436,82.067355,0.205065,3.531868,4.321691,6.195842,12.565927,4.787910,6.380562,41.565980,14.017079
min,0.000000,1896.000000,1.000000,0.000000,13.000000,151.000000,44.000000,0.000000,0.000000,1.000000,1.000000,0.000000
25%,0.000000,1964.000000,5.000000,0.000000,24.000000,173.000000,67.000000,0.000000,0.000000,2.000000,5.000000,0.000000
50%,1.000000,1988.000000,15.000000,0.186047,25.000000,175.000000,70.000000,0.000000,0.000000,5.000000,13.000000,0.000000
75%,1.000000,2004.000000,54.000000,0.364486,27.000000,177.000000,73.000000,3.000000,1.000000,10.000000,36.000000,4.000000
max,1.000000,2016.000000,735.000000,1.000000,67.000000,198.000000,148.000000,230.000000,82.000000,34.000000,270.000000,230.000000


## Handling outliers

In [238]:
exclude_cols = ['noc','season']
def iqr_fun(series:pd.Series,k=3)->tuple[float,float]:
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower,upper

In [239]:
# for col in df.columns.tolist():
#     if col in exclude_cols:
#         continue
#     lower,upper = iqr_fun(df[col])
#     print(f"\nCapping col: {col} min: {df[col].min()} max: {df[col].max()}")
#     print("lower:",lower,"upper:",upper)
#     df[col] = df[col].clip(lower,upper)

## Split into train and test

In [240]:
train,test = train_test_split(df,test_size=0.2,random_state=RANDOM_STATE)

In [241]:
train

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
433,GRE,1,1920,47,0.000000,25.0,176.0,71.0,2.0,1.0,8,34,1
1569,POR,1,1960,65,0.076923,28.0,172.0,71.0,0.0,0.0,11,52,1
371,RUS,1,2000,435,0.445977,26.0,178.0,72.0,63.0,26.0,30,238,89
3148,URS,0,1984,99,0.252525,25.0,175.0,72.0,22.0,10.0,10,38,25
2298,SRB,1,1912,2,0.000000,19.0,172.0,65.0,0.0,0.0,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130,SUI,1,1960,149,0.013423,29.0,175.0,72.0,1.0,0.0,16,90,6
1294,ARM,1,2000,25,0.080000,26.0,175.0,76.0,2.0,1.0,10,26,1
860,TUN,1,2008,26,0.461538,24.0,175.0,70.0,0.0,0.0,10,28,1
3507,SLO,0,1994,22,0.227273,22.0,175.0,70.0,0.0,0.0,3,18,3


In [242]:
test

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
1113,CGO,1,2008,5,0.400000,24.0,177.0,70.0,0.0,0.0,3,5,0
1309,CIV,1,2008,21,0.095238,25.0,179.0,73.0,0.0,0.0,5,7,0
805,RSA,1,1920,39,0.025641,29.0,176.0,70.0,6.0,4.0,7,34,10
1173,GDR,1,1968,226,0.176991,25.0,175.0,70.0,0.0,0.0,18,124,25
2770,VIE,1,1996,6,0.500000,23.0,173.0,67.0,0.0,0.0,4,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3798,TPE,0,1984,12,0.166667,24.0,174.0,73.0,0.0,0.0,5,9,0
376,ARG,1,1900,1,0.000000,23.0,175.0,70.0,0.0,0.0,1,1,0
1263,SWE,1,2016,150,0.573333,28.0,179.0,74.0,8.0,1.0,22,79,11
1075,TJK,1,1996,8,0.250000,22.0,177.0,73.0,0.0,0.0,5,8,0


## Scaling

In [243]:
scaler = RobustScaler()

In [244]:
exclude_cols = ['noc','season','total_medals','athletes_female_percentage']
to_scale_cols = [x for x in df.columns.tolist() if x not in exclude_cols]
to_scale_cols

['year',
 'athletes',
 'avg_age',
 'agv_height',
 'avg_weight',
 'prev_medals',
 'prev_gold_medals',
 'sports',
 'events']

In [245]:
train[to_scale_cols] = scaler.fit_transform(train[to_scale_cols])
train

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
433,GRE,1,-2.000000,0.653061,0.000000,0.000000,0.25,0.166667,0.666667,1.0,0.375,0.733333,1
1569,POR,1,-0.888889,1.020408,0.076923,1.000000,-0.75,0.166667,0.000000,0.0,0.750,1.333333,1
371,RUS,1,0.222222,8.571429,0.445977,0.333333,0.75,0.333333,21.000000,26.0,3.125,7.533333,89
3148,URS,0,-0.222222,1.714286,0.252525,0.000000,0.00,0.333333,7.333333,10.0,0.625,0.866667,25
2298,SRB,1,-2.222222,-0.265306,0.000000,-2.000000,-0.75,-0.833333,0.000000,0.0,-0.500,-0.333333,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130,SUI,1,-0.888889,2.734694,0.013423,1.333333,0.00,0.333333,0.333333,0.0,1.375,2.600000,6
1294,ARM,1,0.222222,0.204082,0.080000,0.333333,0.00,1.000000,0.666667,1.0,0.625,0.466667,1
860,TUN,1,0.444444,0.224490,0.461538,-0.333333,0.00,0.000000,0.000000,0.0,0.625,0.533333,1
3507,SLO,0,0.055556,0.142857,0.227273,-1.000000,0.00,0.000000,0.000000,0.0,-0.250,0.200000,3


In [246]:
test[to_scale_cols] = scaler.transform(test[to_scale_cols])
test

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals
1113,CGO,1,0.444444,-0.204082,0.400000,-0.333333,0.50,0.000000,0.000000,0.0,-0.250,-0.233333,0
1309,CIV,1,0.444444,0.122449,0.095238,0.000000,1.00,0.500000,0.000000,0.0,0.000,-0.166667,0
805,RSA,1,-2.000000,0.489796,0.025641,1.333333,0.25,0.000000,2.000000,4.0,0.250,0.733333,10
1173,GDR,1,-0.666667,4.306122,0.176991,0.000000,0.00,0.000000,0.000000,0.0,1.625,3.733333,25
2770,VIE,1,0.111111,-0.183673,0.500000,-0.666667,-0.50,-0.500000,0.000000,0.0,-0.125,-0.200000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3798,TPE,0,-0.222222,-0.061224,0.166667,-0.333333,-0.25,0.500000,0.000000,0.0,0.000,-0.100000,0
376,ARG,1,-2.555556,-0.285714,0.000000,-0.666667,0.00,0.000000,0.000000,0.0,-0.500,-0.366667,0
1263,SWE,1,0.666667,2.755102,0.573333,1.000000,1.00,0.666667,2.666667,1.0,2.125,2.233333,11
1075,TJK,1,0.111111,-0.142857,0.250000,-1.000000,0.50,0.500000,0.000000,0.0,0.000,-0.133333,0


In [247]:
X_train = train.drop(columns=['noc','total_medals','athletes_female_percentage'])
y_train = train['total_medals']

In [248]:
X_test = test.drop(columns=['noc','total_medals','athletes_female_percentage'])
y_test = test['total_medals']

## Model train and test

In [249]:
lr = LinearRegression()
lr.fit(X_train,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](10,)","[ 0.17,-0.23, 7.94,..., 1.33,-4.81,-1.19]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](10,)","['season','year','athletes',...,'prev_gold_medals','sports','events']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-0.1889
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(10)


In [250]:
lr_predicts = lr.predict(X_test)
lr_predicts[:10]

array([-0.36912226,  0.92432899,  6.82650563, 22.08443844, -0.65199439,
        2.06707301,  0.96778389, 47.52374917,  1.74899283, 40.7542015 ])

In [251]:
xgb = XGBRegressor(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.05,
    random_state=RANDOM_STATE
)
xgb.fit(X_train,y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [252]:
xgb_predicts = xgb.predict(X_test)
xgb_predicts[:10]

array([ 1.3087156e-01,  1.2036531e-01,  4.2817078e+00,  1.8755394e+01,
       -2.4910750e-02,  3.5810950e+00,  2.2188792e-01,  3.2918915e+01,
       -1.0123320e-02,  3.4027657e+01], dtype=float32)

In [253]:
rf = RandomForestRegressor(
    n_estimators=1000,
    max_depth=8,
    random_state=RANDOM_STATE
)
rf.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of 

In [254]:
rf_predicts = rf.predict(X_test)
rf_predicts[:10]

array([ 0.06275899,  0.36531732,  5.04010667, 19.77459944,  0.0633582 ,
        3.31294918,  0.22496617, 35.57728069,  0.37636341, 40.22327036])

## Performance metrics

In [255]:
def metrics(name,y_true,y_pred):
    """Print R², MAE, MSE, RMSE for a model's predictions."""
    r2  = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"\n{name} Performance:")
    print(f"  R²   : {r2:.3f}")       
    print(f"  MAE  : {mae:,.0f}")       
    print(f"  MSE  : {mse:,.0f}")       
    print(f"  RMSE : {rmse:,.0f}")

In [256]:
metrics("LinearRegression",y_test,lr_predicts)
metrics("Xgboost",y_test,xgb_predicts)
metrics("RandomForest",y_test,rf_predicts)


LinearRegression Performance:
  R²   : 0.783
  MAE  : 3
  MSE  : 55
  RMSE : 7

Xgboost Performance:
  R²   : 0.860
  MAE  : 2
  MSE  : 35
  RMSE : 6

RandomForest Performance:
  R²   : 0.833
  MAE  : 2
  MSE  : 42
  RMSE : 6


## Test model

In [257]:
xgb.feature_names_in_

array(['season', 'year', 'athletes', 'avg_age', 'agv_height',
       'avg_weight', 'prev_medals', 'prev_gold_medals', 'sports',
       'events'], dtype='<U16')

In [258]:
data = pd.DataFrame({
    'season':1,
    'year':2000,
    'athletes':24,
    'avg_age':24.0,
    'agv_height':172.0,
    'avg_weight':73.0,
    'prev_medals':19,
    'prev_gold_medals':8,
    'sports':19,'events':32,
},index=[0])
data

,season,year,athletes,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events
0,1,2000,24,24.0,172.0,73.0,19,8,19,32


In [259]:
data[to_scale_cols] = scaler.transform(data[to_scale_cols])
data

,season,year,athletes,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events
0,1,0.222222,0.183673,-0.333333,-0.75,0.5,6.333333,8.0,1.75,0.666667


In [272]:
print("Medals",xgb.predict(data)[0])

Medals 14.357124


In [270]:
print("Medals",lr.predict(data)[0])

Medals 0.7505263372509157


In [271]:
print("Medals",rf.predict(data)[0])

Medals 13.27296792549957
